In [6]:
import warnings
warnings.filterwarnings("ignore")
from tqdm import tqdm
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from datasets import load_from_disk, load_dataset
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision import models
import json


class CNNModel_dropout(nn.Module):
    def __init__(self,num_classes=2, droprate=0.1,epsilon=0.001):
        super(CNNModel_dropout, self).__init__()

        # First convolutional layer: 3 input channels (RGB), 32 filters, 3x3 kernel, padding=1 to maintain spatial size
        
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32, eps=epsilon)

        # Second convolutional layer: 32 input channels, 64 filters
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64, eps=epsilon)

        # Third convolutional layer: 64 input channels, 128 filters
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128, eps=epsilon)

        # Fourth convolutional layer: 128 input channels, 256 filters
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(256, eps=epsilon)

        # Fifth convolutional layer: 256 input channels, 512 filters
        self.conv5 = nn.Conv2d(256, 512, kernel_size=3, padding=1)
        self.bn5 = nn.BatchNorm2d(512, eps=epsilon)
        
        # Max pooling layer: reduces spatial dimensions by half
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Dropout layer to prevent overfitting
        self.drop = nn.Dropout(droprate)
        
        # Global Average Pooling: reduces feature maps to 1x1, turning each channel into a single value
        self.global_avg_pool = nn.AdaptiveAvgPool2d(1)

        # Fully connected layer (Dense layer): Maps 512 features to 'num_classes' outputs
        self.fc = nn.Linear(512, num_classes)
        
    def forward(self, x):
        # First convolution -> BatchNorm -> ReLU -> Max Pooling
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        
        # Second convolution -> BatchNorm -> ReLU -> Max Pooling
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.drop(x) # Apply dropout to prevent overfitting

        # Third convolution -> BatchNorm -> ReLU -> Max Pooling
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = self.drop(x) # Apply dropout
        
        # Fourth convolution -> BatchNorm -> ReLU -> Max Pooling
        x = self.pool(F.relu(self.bn4(self.conv4(x))))
        x = self.drop(x) # Apply dropout
        
        # Fifth convolution -> BatchNorm -> ReLU -> Max Pooling
        x = self.pool(F.relu(self.bn5(self.conv5(x))))        
        x = self.drop(x) # Apply dropout
        
        # Global Average Pooling: Reduces feature maps to 1x1 per channel
        x = self.global_avg_pool(x)
        
        # Flatten the tensor (from [batch_size, 512, 1, 1] to [batch_size, 512])
        x = torch.flatten(x, 1)
        
        # Fully connected layer to get class scores
        x = self.fc(x)
        
        # Apply softmax activation to get class probabilities
        return F.softmax(x, dim=1)

# Custom Dataset class
class DeepFakeDataset(Dataset):
    def __init__(self, dataset, transform=None):
        self.dataset = dataset
        self.transform = transform
        self.start_idx = 4  # Ignore the first 4 rows

    def __len__(self):
        return len(self.dataset['train']) - self.start_idx  # Adjust length

    def __getitem__(self, idx):
        idx += self.start_idx  # Shift index to ignore first 4 rows
        image = self.dataset['train'][idx]['image']  # Assuming it's already a PIL.Image
        label = 0 if idx < 90000 else 1  # First 90k are fake, last 30k are real

        if self.transform:
            image = self.transform(image)
        
        return image, torch.tensor(label, dtype=torch.long)
    
    
def compute_eer(y_true, y_scores):
    """
    Calculate Equal Error Rate (EER)
    :param y_true: True labels (1 for real, 0 for fake)
    :param y_scores: Model scores (probabilities or decision values)
    :return: EER value, threshold where EER occurs
    """
    fpr, tpr, thresholds = roc_curve(y_true, y_scores)
    fnr = 1 - tpr  # False Negative Rate (FRR)
    
    # Find the threshold where FPR ≈ FNR
    eer_threshold = thresholds[np.nanargmin(np.abs(fpr - fnr))]
    eer = fpr[np.nanargmin(np.abs(fpr - fnr))]
    
    return eer, eer_threshold

def ensure_three_channels(image):
    """
    Convert all images to 3-channel format.
    - If the image is grayscale (1 channel), convert it to RGB (3 channels).
    - If it's already RGB, keep it unchanged.
    """
    if image.mode != "RGB":  # Check if the image is not already RGB
        image = image.convert("RGB")  # Convert grayscale (1 channel) to RGB (3 channels)
    return image


def reset_weights(m):
    """
    Reset the weights of the model.
    This function is called to reinitialize the model's parameters.
    """
    if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
        m.reset_parameters()  # Reset parameters for Conv2d and Linear layers	

# Define a transformation
transform_normal = transforms.Compose([
    transforms.Lambda(ensure_three_channels),
    transforms.Resize((224, 224)),  # Resize to match CNN input size
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

def train_model(model,full_dataset,results,model_name,k_folds = 5,num_epochs = 20,lr=0.0001):
    model.apply(reset_weights)
    results[model_name] = {}
    results[model_name]['kfolds'] = {}
    # Move model to GPU if available
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    assert next(model.parameters()).is_cuda, "Model is NOT on GPU!"

    # Loss function & optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)

    all_accuracies = []
    all_aucs = []
    all_eers = []
    
    for fold, (train_idx, test_idx) in enumerate(kf.split(full_dataset)):
        results[model_name]['kfolds'][fold+1] = {}
        print(f'Fold {fold+1}/{k_folds}')
        
        train_subset = torch.utils.data.Subset(full_dataset, train_idx.tolist())
        test_subset = torch.utils.data.Subset(full_dataset, test_idx.tolist())
        train_loader = DataLoader(train_subset, batch_size=64, shuffle=True)
        test_loader = DataLoader(test_subset, batch_size=64, shuffle=False)
        for epoch in range(num_epochs):
            results[model_name]['kfolds'][fold+1][epoch] = {}
            model.train()
            running_loss = 0.0
            with tqdm(total=len(train_loader), desc=f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}") as pbar:
                for images, labels in train_loader:
                    images, labels = images.to(device), labels.to(device)
                    optimizer.zero_grad()
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                    loss.backward()
                    optimizer.step()

                    running_loss += loss.item()
                
                    pbar.set_description(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}")
                    pbar.update(1)    
            results[model_name]['kfolds'][fold+1][epoch]['loss'] =  "{:.4f}".format(running_loss / len(train_loader)) 
            print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}")
        
        # Evaluation
        model.eval()
        all_preds, all_labels = [], []
        
        with torch.no_grad():
            with tqdm(total=len(test_loader), desc=f"Evaluating") as pbar:
                for images, labels in test_loader:
                    images, labels = images.to(device), labels.to(device)
                    outputs = model(images)
                    probabilities = torch.softmax(outputs, dim=1)[:, 1]  
                    all_preds.extend(probabilities.cpu().numpy())
                    all_labels.extend(labels.cpu().numpy())
                    pbar.update(1) 

        all_preds = np.array(all_preds)
        all_labels = np.array(all_labels)

        # ---- Compute Metrics ----
        accuracy = accuracy_score(all_labels, all_preds > 0.5)
        auc = roc_auc_score(all_labels, all_preds)
        eer_value, eer_threshold = compute_eer(all_labels, all_preds)

        all_accuracies.append(accuracy)
        all_aucs.append(auc)
        all_eers.append(eer_value)

        results[model_name]['kfolds'][fold+1]['Accuracy'] = "{:.4f}".format(accuracy) 
        results[model_name]['kfolds'][fold+1]['AUC'] = "{:.4f}".format(auc) 
        results[model_name]['kfolds'][fold+1]['EER'] = "{:.4f}".format(eer_value) 
        results[model_name]['kfolds'][fold+1]['Threshold'] = "{:.4f}".format(eer_threshold) 

        print(f"Fold {fold+1} - Accuracy: {accuracy:.4f}, AUC: {auc:.4f}, EER: {eer_value:.4f} (Threshold: {eer_threshold:.4f})")
    results[model_name]['Avg_Accuracy'] = "{:.4f}".format(np.mean(all_accuracies)) 
    results[model_name]['Avg_AUC'] = "{:.4f}".format(np.mean(all_aucs)) 
    results[model_name]['Avg_EER'] = "{:.4f}".format(np.mean(all_eers)) 
    print("Training complete.")
    print(f"Average Accuracy: {np.mean(all_accuracies):.4f}")
    print(f"Average AUC: {np.mean(all_aucs):.4f}")
    print(f"Average EER: {np.mean(all_eers):.4f}")
    return model , results


In [7]:
save_models_dir = "./models/"
save_dataset_dir = "./dataset/"
results = {}

In [8]:
try:
    dataset = load_from_disk("./dataset")
    print("Dataset loaded successfully!")
except FileNotFoundError:
    print("Dataset not found. Trying to download and save him")
    dataset = load_dataset("OpenRL/DeepFakeFace")
    dataset.save_to_disk(save_dataset_dir) 
print(dataset)

dataset = DeepFakeDataset(dataset, transform=transform_normal)


Dataset loaded successfully!
DatasetDict({
    train: Dataset({
        features: ['image'],
        num_rows: 120004
    })
})


In [9]:
full_dataset = dataset
train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(full_dataset, [train_size, test_size])

In [10]:
if __name__ == "__main__":
    model = CNNModel_dropout()
    model , results = train_model(model,full_dataset,results,'CNN_dropout_normal',k_folds = 5,num_epochs = 20,lr=0.0001)

Fold 1/5


Epoch [1/20], Loss: 0.5479: 100%|██████████| 1500/1500 [07:27<00:00,  3.35it/s]


Epoch [1/20], Loss: 0.5479


Epoch [2/20], Loss: 0.5109: 100%|██████████| 1500/1500 [07:30<00:00,  3.33it/s]


Epoch [2/20], Loss: 0.5109


...

Epoch [19/20], Loss: 0.3179


Epoch [20/20], Loss: 0.3180: 100%|██████████| 1500/1500 [06:55<00:00,  3.61it/s]


Epoch [20/20], Loss: 0.3180


Evaluating: 100%|██████████| 375/375 [01:26<00:00,  4.32it/s]

Fold 5 - Accuracy: 0.9920, AUC: 0.9997, EER: 0.0071 (Threshold: 0.2801)
Training complete.
Average Accuracy: 0.9593
Average AUC: 0.9755
Average EER: 0.0529


In [11]:
os.makedirs(save_models_dir, exist_ok=True)
torch.save(model, os.path.join(save_models_dir, "CNN_dropout_normal_fine_tuned.pth"))
with open("results.json", "w") as json_file: 
    json.dump(results, json_file, indent=4)